<div><h1>Async Programming in Python</h1><img src="images/65ca262bc1cfe27d0d587d2a_4177744_d725_4.jpg"></img></div>

<div><h2>Introduction and Basic Concepts</h2><p>Async (asynchronous) programming in Python is a programming paradigm that allows you to manage I/O-bound operations (such as network requests, file access, and similar tasks) more efficiently.</p><h5>Key Concepts:</h5><p>1) Synchronous: Code is executed line by line; each line must finish before the next one starts.<br>2) Asynchronous: Multiple tasks can overlap without waiting for each task to complete before starting another.<br>3) I/O-bound: Operations whose execution time depends more on input/output waiting than CPU processing.</p></div>

<div><h3>Async Architecture in Python</h3><h5>Event Loop</h5><p>The heart of async programming in Python. The Event Loop is responsible for:</p><ul><li>Scheduling coroutine execution</li><li>Managing callbacks</li><li>Performing I/O operations</li><li>Running functions and subroutines</li></ul><p>Let us look at a simple Event Loop example:</p></div>

In [1]:
import asyncio

async def main():
    print('Hello')
    await asyncio.sleep(1)
    print('World')

# asyncio.run(main())  # For creating Event Loop
await main()

Hello
World


<div><h5>Coroutines</h5><p>Coroutines in Python are objects that contain a set of instructions and can be executed whenever needed. Python provides two types of coroutines:</p><p>1) Classic Coroutines: Created using generator functions defined with the <b>yield</b> keyword.<br>2) Async Coroutines: The type used in modern async programming, which we discuss here.</p><ul><li>Defined using the <b>async def</b> keyword.</li><li>Can transfer control to another coroutine using <b>await</b>.</li><li>They are not ordinary functions; they are objects that must be explicitly awaited.</li></ul><p>Here is an example for better understanding:</p></div>

In [2]:
import asyncio

async def task1():
    print("Task 1 started")
    await asyncio.sleep(3)
    print("Task 1 finished")

async def task2():
    print("Task 2 started")
    await asyncio.sleep(1)
    print("Task 2 finished")

async def main():
    await asyncio.gather(task1(), task2())  # Runs both tasks together

# asyncio.run(main())
await main()

Task 1 started
Task 2 started
Task 2 finished
Task 1 finished


<div><p>First, we create an Event Loop using <b>asyncio.run()</b> and delegate control of that Event Loop to the async function <b>main</b>. Using <b>asyncio.gather()</b>, we can execute two or more coroutines concurrently. Here, we run <b>task1</b> and <b>task2</b> concurrently. Since asyncio functions are asynchronous, we use <b>await</b> when calling them. Inside task1 and task2 we use <b>asyncio.sleep()</b>. Why not use <b>time.sleep()</b>? Because time.sleep() is a synchronous function. When it runs, the Event Loop loses control until the function finishes. If we did not use the async version of sleep, task1 and task2 would not be able to run concurrently.</p></div>

<div><h4>Async Execution Model in Python</h4><h5>Cooperative Multitasking</h5><p>Unlike multithreading, where the operating system performs scheduling, in async programming:</p><ul><li>Each coroutine voluntarily yields control using <b>await</b>.</li><li>No coroutine can forcibly interrupt another; each coroutine runs independently.</li><li>All coroutines must cooperate.</li></ul><h5>How It Works:</h5><p>1) The Event Loop starts running.<br>2) Tasks are created and added to the Event Loop.<br>3) Each Task runs until it reaches an <b>await</b> statement.<br>4) Control returns to the Event Loop.<br>5) Steps 2–4 may repeat many times.<br>6) Eventually, the Event Loop finishes.</p><img src="images/1_mhopIIMmI5SZgGHQXZTbwQ.webp"></img><p>Next, we examine the concept of a Task in more detail.</p></div>

<div><h4>The Task Concept and asyncio.create_task()</h4><p>In Python async programming, a Task is a fundamental concept representing the execution of a coroutine within the event loop.</p><p><b>What is a Task?</b></p><ul><li>An execution unit in the event loop.</li><li>A subclass of <b>Future</b> that wraps a coroutine.</li><li>Provides the ability to track a coroutine’s execution state.</li><li>Can run in the background without being awaited immediately.</li></ul><h5>asyncio.create_task()</h5><p>This function takes a coroutine, converts it into a <b>Task</b>, and immediately schedules it for execution in the event loop.</p><p>How it works:</p></div>

In [3]:
import asyncio

async def my_coroutine():
    await asyncio.sleep(1)
    return "Done"

# Convert a coroutine to a task
task = asyncio.create_task(my_coroutine())
print(await task) # Result

Done


<div><p>Important Features:</p><p>1) Concurrent execution: A Task starts running immediately (it does not require an immediate await).<br>2) Traceable: Its status can be inspected.<br>3) Awaitable: Its result can be obtained later using await.</p><p>Next, we look at a deeper example for better understanding:</p></div>

In [1]:
import asyncio

async def fetch_data(id, delay):
    print(f"Task {id} starting, will take {delay} seconds")
    await asyncio.sleep(delay)
    print(f"Task {id} completed")
    return f"Result from task {id}"

async def main():
    # Create three task
    task1 = asyncio.create_task(fetch_data(1, 2))
    task2 = asyncio.create_task(fetch_data(2, 1))
    task3 = asyncio.create_task(fetch_data(3, 3))

    # We can do something else when tasks are running ...
    print("Tasks are running in the background...")

    # Get results of the tasks
    results = await asyncio.gather(task1, task2, task3)
    print("All tasks done:", results)

# asyncio.run(main())
await main()

Tasks are running in the background...
Task 1 starting, will take 2 seconds
Task 2 starting, will take 1 seconds
Task 3 starting, will take 3 seconds
Task 2 completed
Task 1 completed
Task 3 completed
All tasks done: ['Result from task 1', 'Result from task 2', 'Result from task 3']


<div><p>Inside the main function, we use the <b>fetch_data</b> coroutine and <b>create_task</b> to create three different Tasks. These tasks are scheduled in the event loop and run concurrently. While these tasks are running in the background, the event loop still has control and can continue performing other work. Finally, we can retrieve task results individually using <b>await</b>, or collect them all at once as a list using <b>asyncio.gather()</b>.</p></div>

<div><h5>Key Notes About Tasks</h5><p>1) Automatic scheduling: When a Task is created, it is immediately scheduled for execution.<br>2) Non-blocking: Creating a Task does not stop the execution flow (unlike directly awaiting a coroutine).<br>3) Group management: Multiple Tasks can be managed using <b>asyncio.gather()</b> or <b>asyncio.wait()</b>.<br>4) Cancellation: A Task can be canceled using the <b>cancel()</b> method.<br>5) Status inspection: The <b>done()</b> method can be used to check whether a Task has finished, is still running, or is in another state.</p><h5>Summary</h5><ul><li>A Task is a wrapper that enables concurrent coroutine execution.</li><li>create_task converts a coroutine into a Task and starts it immediately.</li><li>You can create multiple Tasks and collect their results later.</li><li>Tasks provide better management of concurrent workflows.</li></ul><p>This mechanism is the core of asynchronous programming in Python and allows efficient handling of I/O-bound operations.</p></div>